## Configuration

Change only this cell to switch between runs.

MODEL_DIR    = Path("../models/bert_biomedbert_re_A0_fixed").resolve()
POOLING_MODE = "marker-only"

# A2
MODEL_DIR    = Path("../models/bert_biomedbert_re_A2_weighted_fixed").resolve()
POOLING_MODE = "marker-only"

# A3
MODEL_DIR    = Path("../models/bert_biomedbert_re_A3_mentionmean").resolve()
POOLING_MODE = "mention-mean"

# A4
MODEL_DIR    = Path("../models/bert_biomedbert_re_A4_labelsmooth").resolve()
POOLING_MODE = "mention-mean"

# A5
MODEL_DIR    = Path("../models/bert_biomedbert_re_A5_hardneg").resolve()
POOLING_MODE = "mention-mean"

In [1]:
from pathlib import Path
import os, json, math, random, re, itertools
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

# ════════════════════════════════════════════════════════════
# CHANGE THESE TWO LINES TO SWITCH RUN
# ════════════════════════════════════════════════════════════
MODEL_DIR    = Path("models/biolink_bert_re_A5_hardneg").resolve()
POOLING_MODE = "mention-mean"                        # ✅ come nel training
BASE_MODEL   = "michiyasunaga/BioLinkBERT-base"
# Forza checkpoint specifico (None = usa load_best / last)
#FORCE_CHECKPOINT = "checkpoint-12000"  # miglior micro F1 in training

#
# ════════════════════════════════════════════════════════════

DEV_PATH   = Path("../../data/GutBrainIE_Full_Collection_2026/Annotations/Dev/json_format/dev.json")


BATCH_SIZE   = 16
MAX_LENGTH   = 512
WINDOW_CHARS = 300

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("MODEL_DIR   :", MODEL_DIR)
print("POOLING_MODE:", POOLING_MODE)
print("DEVICE      :", DEVICE)
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")


C:\Users\super\Documents\UniPd\ATA\GutBrainIE\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MODEL_DIR   : C:\Users\super\Documents\UniPd\ATA\GutBrainIE\src\re\models\biolink_bert_re_A5_hardneg
POOLING_MODE: mention-mean
DEVICE      : cuda
GPU         : NVIDIA GeForce RTX 5070 Laptop GPU


## Labels and Legal Relation Pairs

In [2]:
LEGAL_ENTITY_LABELS = {
    "anatomical location","animal","bacteria","biomedical technique","chemical","DDF",
    "dietary supplement","drug","food","gene","human","microbiome","statistical technique"
}
LEGAL_RELATION_LABELS = {
    "administered","affect","change abundance","change effect","change expression","compared to",
    "impact","influence","interact","is a","is linked to","located in","part of","produced by",
    "strike","target","used by"
}
RELATION_LABELS = [
    "no relation","administered","affect","change abundance","change effect","change expression",
    "compared to","impact","influence","interact","is a","is linked to","located in","part of",
    "produced by","strike","target","used by"
]
label2id = {l: i for i, l in enumerate(RELATION_LABELS)}
id2label = {i: l for i, l in enumerate(RELATION_LABELS)}

def norm_ent(label):
    if label is None: return ""
    lab = str(label).strip()
    return "DDF" if lab.lower() == "ddf" else lab

def norm_span(s):
    return re.sub(r"\s+", " ", str(s).strip())

LEGAL_RELATIONS = [
    ("DDF","affect","DDF"),("microbiome","is linked to","DDF"),("DDF","target","human"),
    ("drug","change effect","DDF"),("DDF","is a","DDF"),("microbiome","located in","human"),
    ("chemical","influence","DDF"),("dietary supplement","influence","DDF"),("DDF","target","animal"),
    ("chemical","impact","microbiome"),("anatomical location","located in","animal"),
    ("microbiome","located in","animal"),("chemical","located in","anatomical location"),
    ("bacteria","part of","microbiome"),("DDF","strike","anatomical location"),
    ("drug","administered","animal"),("bacteria","influence","DDF"),("drug","impact","microbiome"),
    ("DDF","change abundance","microbiome"),("microbiome","located in","anatomical location"),
    ("microbiome","used by","biomedical technique"),("chemical","produced by","microbiome"),
    ("dietary supplement","impact","microbiome"),("bacteria","located in","animal"),
    ("animal","used by","biomedical technique"),("chemical","impact","bacteria"),
    ("chemical","located in","animal"),("food","impact","bacteria"),
    ("microbiome","compared to","microbiome"),("human","used by","biomedical technique"),
    ("bacteria","change expression","gene"),("chemical","located in","human"),
    ("drug","interact","chemical"),("food","administered","human"),
    ("DDF","change abundance","bacteria"),("chemical","interact","chemical"),
    ("chemical","part of","chemical"),("dietary supplement","impact","bacteria"),
    ("DDF","interact","chemical"),("food","impact","microbiome"),("food","influence","DDF"),
    ("bacteria","located in","human"),("dietary supplement","administered","human"),
    ("bacteria","interact","chemical"),("drug","change expression","gene"),
    ("drug","impact","bacteria"),("drug","administered","human"),
    ("anatomical location","located in","human"),("dietary supplement","change expression","gene"),
    ("chemical","change expression","gene"),("bacteria","interact","bacteria"),
    ("drug","interact","drug"),("microbiome","change expression","gene"),
    ("bacteria","interact","drug"),("food","change expression","gene"),
]
legal_pairs = {}
for s, p, o in LEGAL_RELATIONS:
    legal_pairs.setdefault((norm_ent(s), norm_ent(o)), set()).add(p)

print(f"Relation labels: {len(RELATION_LABELS)}")
print(f"Legal type pairs: {len(legal_pairs)}")


Relation labels: 18
Legal type pairs: 52


## Model Architecture

Supports both `marker-only` (A0, A2) and `mention-mean` (A3, A4, A5) pooling.
Set via `POOLING_MODE` in the config cell.

In [3]:
def entity_average(hidden, mask):
    """Mean pooling over entity mention tokens."""
    mask = mask.unsqueeze(-1).float()
    summed = (hidden * mask).sum(dim=1)
    count = mask.sum(dim=1).clamp(min=1e-6)
    return summed / count

class BertForREWithEntityMarkers(nn.Module):
    def __init__(self, model_name, num_labels, pooling_mode="marker-only"):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.bert.config.hidden_size * 2, num_labels)
        self.num_labels = num_labels
        self.pooling_mode = pooling_mode

    def forward(self, input_ids, attention_mask, e1_mask, e2_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        seq = outputs.last_hidden_state
        if self.pooling_mode == "mention-mean":
            e1_h = entity_average(seq, e1_mask)
            e2_h = entity_average(seq, e2_mask)
        else:  # marker-only
            e1_h = torch.bmm(e1_mask.unsqueeze(1).float(), seq).squeeze(1)
            e2_h = torch.bmm(e2_mask.unsqueeze(1).float(), seq).squeeze(1)
        concat_h = self.dropout(torch.cat([e1_h, e2_h], dim=-1))
        logits = self.classifier(concat_h)
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits.view(-1, self.num_labels), labels.view(-1))
        return {"loss": loss, "logits": logits}

print("Model class defined — pooling mode will be:", POOLING_MODE)


Model class defined — pooling mode will be: mention-mean


## Load Model and Tokenizer

In [4]:
def find_last_checkpoint(root_dir: Path) -> Path:
    ckpts = [d for d in root_dir.iterdir() if d.is_dir() and d.name.startswith("checkpoint-")]
    if not ckpts:
        return root_dir
    return sorted(ckpts, key=lambda x: int(x.name.split("-")[1]))[-1]

if not MODEL_DIR.exists():
    raise FileNotFoundError(f"Model directory not found: {MODEL_DIR}")

# if FORCE_CHECKPOINT:
#     LOAD_DIR = MODEL_DIR / FORCE_CHECKPOINT
# else:
#    LOAD_DIR = find_last_checkpoint(MODEL_DIR)
LOAD_DIR = find_last_checkpoint(MODEL_DIR)
STATE_PATH = LOAD_DIR / "pytorch_model.bin"
print("LOAD_DIR  :", LOAD_DIR)
print("STATE_PATH:", STATE_PATH)

if not STATE_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found: {STATE_PATH}")

# Tokenizer — caricato dalla cartella del modello (include special tokens)
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), use_fast=True, local_files_only=True)
e1_token_id = tokenizer.convert_tokens_to_ids("[E1]")
e2_token_id = tokenizer.convert_tokens_to_ids("[E2]")
print(f"Tokenizer loaded — vocab size: {len(tokenizer)}")
print(f"[E1] id: {e1_token_id}  |  [E2] id: {e2_token_id}")

# Modello
model = BertForREWithEntityMarkers(BASE_MODEL, num_labels=len(RELATION_LABELS), pooling_mode=POOLING_MODE)
model.bert.resize_token_embeddings(len(tokenizer))
state_dict = torch.load(STATE_PATH, map_location="cpu")
model.load_state_dict(state_dict)
model.to(DEVICE)
model.eval()
print(f"Model loaded on {DEVICE} — pooling: {POOLING_MODE}")
print("hidden_size:", model.bert.config.hidden_size)
print("num_layers:", model.bert.config.num_hidden_layers)
print("classifier weight shape:", model.classifier.weight.shape)

LOAD_DIR  : C:\Users\super\Documents\UniPd\ATA\GutBrainIE\src\re\models\biolink_bert_re_A5_hardneg\checkpoint-15528
STATE_PATH: C:\Users\super\Documents\UniPd\ATA\GutBrainIE\src\re\models\biolink_bert_re_A5_hardneg\checkpoint-15528\pytorch_model.bin
Tokenizer loaded — vocab size: 28899
[E1] id: 28895  |  [E2] id: 28897


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16583.22it/s]
BertModel LOAD REPORT from: michiyasunaga/BioLinkBERT-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model loaded on cuda — pooling: mention-mean
hidden_size: 768
num_layers: 12
classifier weight shape: torch.Size([18, 1536])


## Load Dev Data

In [5]:
def load_re_data(file_paths):
    all_data = {}
    for p in file_paths:
        if os.path.exists(p):
            with open(p, "r", encoding="utf-8") as f:
                data = json.load(f)
            all_data.update(data)
            print(f"Loaded {len(data)} docs from {os.path.basename(p)}")
    return all_data

def create_full_text_with_offsets(title, abstract):
    full_text = f"{title} {abstract}"
    return full_text, len(title) + 1

def adjust_entity_positions(entity, abstract_offset):
    if entity["location"] == "abstract":
        return {**entity,
                "start_idx": entity["start_idx"] + abstract_offset,
                "end_idx":   entity["end_idx"]   + abstract_offset}
    return dict(entity)

def insert_entity_markers(text, subject, obj):
    entities = sorted([
        (subject["start_idx"], subject["end_idx"], "[E1]", "[/E1]"),
        (obj["start_idx"],     obj["end_idx"],     "[E2]", "[/E2]"),
    ], key=lambda x: x[0])
    marked, offset = text, 0
    for start, end, sm, em in entities:
        a, b = start + offset, end + offset + 1
        marked = marked[:a] + sm + marked[a:b] + em + marked[b:]
        offset += len(sm) + len(em)
    return marked

def build_window_around_entities(text, subject, obj, window_chars=300):
    left  = min(subject["start_idx"], obj["start_idx"])
    right = max(subject["end_idx"],   obj["end_idx"])
    ws = max(0, left - window_chars)
    we = min(len(text) - 1, right + window_chars)
    win = text[ws:we + 1]
    sw = {**subject, "start_idx": subject["start_idx"] - ws, "end_idx": subject["end_idx"] - ws}
    ow = {**obj,     "start_idx": obj["start_idx"] - ws,     "end_idx": obj["end_idx"] - ws}
    return win, sw, ow

def build_marked_text(text, subj, obj, window_chars=300):
    w, sw, ow = build_window_around_entities(text, subj, obj, window_chars)
    return insert_entity_markers(w, sw, ow)

def gold_tuple(r):
    return (norm_span(r["subject_text_span"]), norm_ent(r["subject_label"]),
            r["predicate"].strip(),
            norm_span(r["object_text_span"]),  norm_ent(r["object_label"]))

def build_gold_maps(dev_data):
    gold = {}
    for pmid, art in dev_data.items():
        s = set()
        for r in art.get("mention_level_relations", []):
            if r["predicate"].strip() in LEGAL_RELATION_LABELS:
                s.add(gold_tuple(r))
        gold[str(pmid)] = s
    return gold

def micro_scores(gold, pred):
    tp = fp = fn = 0
    for pmid, g in gold.items():
        p = pred.get(pmid, set())
        tp += len(g & p); fp += len(p - g); fn += len(g - p)
    P = tp/(tp+fp) if (tp+fp) else 0.0
    R = tp/(tp+fn) if (tp+fn) else 0.0
    F1 = 2*P*R/(P+R) if (P+R) else 0.0
    return {"P": round(P,4), "R": round(R,4), "F1": round(F1,4), "TP": tp, "FP": fp, "FN": fn}

def macro_scores(gold, pred):
    preds = sorted({t[2] for s in gold.values() for t in s} | {t[2] for s in pred.values() for t in s})
    vals = []
    for pr in preds:
        tp=fp=fn=0
        for pmid, g in gold.items():
            gp = {t for t in g if t[2]==pr}
            pp = {t for t in pred.get(pmid, set()) if t[2]==pr}
            tp+=len(gp&pp); fp+=len(pp-gp); fn+=len(gp-pp)
        P = tp/(tp+fp) if (tp+fp) else 0.0
        R = tp/(tp+fn) if (tp+fn) else 0.0
        vals.append((P, R, 2*P*R/(P+R) if (P+R) else 0.0))
    return {"macro_P": round(float(np.mean([x[0] for x in vals])),4) if vals else 0.0,
            "macro_R": round(float(np.mean([x[1] for x in vals])),4) if vals else 0.0,
            "macro_F1": round(float(np.mean([x[2] for x in vals])),4) if vals else 0.0}

dev_data   = load_re_data([DEV_PATH])
gold_by_doc = build_gold_maps(dev_data)
print(f"Dev docs: {len(dev_data)}  |  Gold relations: {sum(len(v) for v in gold_by_doc.values())}")


Loaded 80 docs from dev.json
Dev docs: 80  |  Gold relations: 1139


## Cache Dev Logits

Computes logits for all candidate pairs on the dev set once and caches to disk.
On subsequent runs, loads from cache — no GPU needed.

In [6]:
@torch.no_grad()
def cache_dev_logits_once(model, tokenizer, dev_data, legal_pairs,
                          e1_token_id, e2_token_id, device,
                          batch_size=16, max_length=512, window_chars=300, cache_path=None):
    if cache_path and os.path.exists(cache_path):
        print("[cache] loading:", cache_path)
        return torch.load(cache_path, map_location="cpu")

    model.eval()
    cache = {}

    for pmid, article in tqdm(dev_data.items(), desc="Caching logits"):
        title    = article["metadata"]["title"]
        abstract = article["metadata"]["abstract"]
        full_text, abstract_offset = create_full_text_with_offsets(title, abstract)

        adjusted = [{
            **adjust_entity_positions(e, abstract_offset),
            "label":     norm_ent(e["label"]),
            "text_span": norm_span(e["text_span"]),
        } for e in article["entities"]]

        pair_examples, rows_meta = [], []
        for i, subj in enumerate(adjusted):
            for j, obj in enumerate(adjusted):
                if i == j: continue
                if (subj["label"], obj["label"]) not in legal_pairs: continue
                pair_examples.append((full_text, subj, obj))
                rows_meta.append({
                    "k":   (subj["text_span"], subj["label"], obj["text_span"], obj["label"]),
                    "dist": abs(subj["start_idx"] - obj["start_idx"]),
                    "subject_label": subj["label"],
                    "object_label":  obj["label"],
                })

        if not pair_examples:
            cache[str(pmid)] = []
            continue

        marked_texts = [build_marked_text(t, s, o, window_chars) for t, s, o in pair_examples]
        rows, idx = [], 0

        for start in range(0, len(marked_texts), batch_size):
            batch = marked_texts[start:start+batch_size]
            enc = tokenizer(batch, truncation=True, max_length=max_length,
                            padding=True, return_tensors="pt")
            input_ids      = enc["input_ids"].to(device)
            attention_mask = enc["attention_mask"].to(device)
            e1_mask = (input_ids == e1_token_id).long()
            e2_mask = (input_ids == e2_token_id).long()

            logits = model(input_ids=input_ids, attention_mask=attention_mask,
                           e1_mask=e1_mask, e2_mask=e2_mask)["logits"]
            logits = logits.detach().cpu().float()

            for b in range(logits.shape[0]):
                row = dict(rows_meta[idx])
                row["logits"] = logits[b]
                rows.append(row)
                idx += 1

        cache[str(pmid)] = rows

    if cache_path:
        torch.save(cache, cache_path)
        print("[cache] saved:", cache_path)
    return cache

# Cache salvata dentro la cartella del modello — specifica per run
LOGITS_CACHE_PATH = str(MODEL_DIR / "dev_logits_cache.pt")

dev_cache = cache_dev_logits_once(
    model=model, tokenizer=tokenizer, dev_data=dev_data,
    legal_pairs=legal_pairs, e1_token_id=e1_token_id, e2_token_id=e2_token_id,
    device=DEVICE, batch_size=BATCH_SIZE, max_length=MAX_LENGTH,
    window_chars=WINDOW_CHARS, cache_path=LOGITS_CACHE_PATH,
)
print(f"Cached docs: {len(dev_cache)}  |  Total pairs: {sum(len(v) for v in dev_cache.values())}")


Caching logits: 100%|██████████| 80/80 [06:59<00:00,  5.24s/it]


[cache] saved: C:\Users\super\Documents\UniPd\ATA\GutBrainIE\src\re\models\biolink_bert_re_A5_hardneg\dev_logits_cache.pt
Cached docs: 80  |  Total pairs: 50202


## Predicate-Specific Decoding

Same logic as the original best inference — per-predicate threshold tuning.

In [7]:
PREDICATE_PRIOR = {"strike": 0.85, "used by": 0.90, "part of": 0.90}

def apply_predicate_prior(pred, prob):
    return prob * PREDICATE_PRIOR.get(pred, 1.0)

def softmax_np(x):
    x = x - np.max(x)
    ex = np.exp(x)
    return ex / ex.sum()

def distance_min_prob(distance):
    if distance < 150:  return 0.10
    elif distance < 300: return 0.20
    else:               return 0.30

def row_to_legal_scores(row, legal_pairs, label2id, id2label, temperature=1.0, renorm_legal=True):
    s_lab, o_lab = row["subject_label"], row["object_label"]
    allowed_preds = sorted(legal_pairs.get((s_lab, o_lab), []))
    if not allowed_preds:
        return None

    logits = row["logits"].numpy() / temperature

    if renorm_legal:
        allowed_ids = [label2id["no relation"]] + [label2id[p] for p in allowed_preds]
        probs_sub = softmax_np(logits[allowed_ids])
        p_no = float(probs_sub[0])
        rel_probs = probs_sub[1:]
        best_idx = int(np.argmax(rel_probs))
        pred_label = allowed_preds[best_idx]
        p_best = float(rel_probs[best_idx])
        p_second = sorted(rel_probs.tolist(), reverse=True)[1] if len(rel_probs) > 1 else 0.0
    else:
        probs = softmax_np(logits)
        p_no  = float(probs[label2id["no relation"]])
        legal_probs = sorted([(id2label[label2id[p]], float(probs[label2id[p]])) for p in allowed_preds],
                              key=lambda x: x[1], reverse=True)
        pred_label, p_best = legal_probs[0]
        p_second = legal_probs[1][1] if len(legal_probs) > 1 else 0.0

    p_best_adj = apply_predicate_prior(pred_label, p_best)
    return {"pred_label": pred_label, "p_best": p_best, "p_best_adj": p_best_adj,
            "p_second": p_second, "p_no": p_no,
            "margin_val": p_best - p_no, "top2_gap": p_best - p_second}

def decode_doc_pred_specific(rows, legal_pairs, label2id, id2label,
                              default_max_chars, default_min_prob, default_margin, default_top2_gap,
                              temperature, renorm_legal,
                              max_chars_by_pred=None, min_prob_by_pred=None,
                              margin_by_pred=None, top2_gap_by_pred=None,
                              use_distance_aware_prob=True):
    best_for_pair = {}
    for row in rows:
        scores = row_to_legal_scores(row, legal_pairs, label2id, id2label, temperature, renorm_legal)
        if scores is None: continue
        pred = scores["pred_label"]
        maxc  = (max_chars_by_pred  or {}).get(pred, default_max_chars)
        minp  = (min_prob_by_pred   or {}).get(pred, default_min_prob)
        marg  = (margin_by_pred     or {}).get(pred, default_margin)
        top2g = (top2_gap_by_pred   or {}).get(pred, default_top2_gap)
        if row["dist"] > maxc: continue
        effective_minp = max(minp, distance_min_prob(row["dist"])) if use_distance_aware_prob else minp
        if scores["p_best_adj"] < effective_minp: continue
        if scores["margin_val"] < marg:           continue
        if scores["top2_gap"]   < top2g:          continue
        k = row["k"]
        prev = best_for_pair.get(k)
        if prev is None or scores["p_best_adj"] > prev[1]:
            best_for_pair[k] = (pred, scores["p_best_adj"])
    out = set()
    for (st, sl, ot, ol), (pred, score) in best_for_pair.items():
        out.add((st, sl, pred, ot, ol))
    return out, best_for_pair

def build_predictions_json(best_for_pair):
    return [{"subject_text_span": st, "subject_label": sl, "predicate": pred,
             "object_text_span": ot, "object_label": ol}
            for (st, sl, ot, ol), (pred, _) in sorted(best_for_pair.items())]

print("Decoding functions defined")


Decoding functions defined


## Best Global Hyperparameters

From the original tuning on A2. These are kept fixed; per-predicate tuning refines on top.

In [8]:
BEST_MAX    = 250
BEST_MINP   = 0.10
BEST_MARG   = 0.25
BEST_TEMP   = 1.25
BEST_RENORM = True

PREDICATES_IN_DEV = sorted({t[2] for s in gold_by_doc.values() for t in s})
PRED_MARGIN_GRID   = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25]  # aggiungi 0.00
PRED_MINPROB_GRID  = [0.00, 0.05, 0.10, 0.15, 0.20]         # aggiungi 0.05
PRED_MAXCHARS_GRID = [100, 150, 200, 250, 300, 400, 500]
PRED_TOP2_GAP_GRID = [0.00, 0.02, 0.04, 0.06, 0.08]

# Ottimizza i global defaults su micro F1 (metrica ufficiale GutBrainIE 2026)
print("Tuning global defaults on micro F1...")
best_global = None
for marg in [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:
    for minp in [0.00, 0.05, 0.10, 0.15, 0.20]:
        for temp in [1.0, 1.25, 1.5]:
            pred_by_doc = {}
            for pmid, rows in dev_cache.items():
                pred_set, _ = decode_doc_pred_specific(
                    rows, legal_pairs, label2id, id2label,
                    BEST_MAX, minp, marg, 0.0, temp, BEST_RENORM,
                    use_distance_aware_prob=True)
                pred_by_doc[pmid] = pred_set
            sc = micro_scores(gold_by_doc, pred_by_doc)
            if best_global is None or sc["F1"] > best_global["F1"]:
                best_global = {"marg": marg, "minp": minp, "temp": temp, **sc}

BEST_MARG = best_global["marg"]
BEST_MINP = best_global["minp"]
BEST_TEMP = best_global["temp"]
print(f"Best global: margin={BEST_MARG} minp={BEST_MINP} temp={BEST_TEMP}")
print(f"  Micro F1={best_global['F1']:.4f}  P={best_global['P']:.4f}  R={best_global['R']:.4f}")
print("Predicates in dev:", PREDICATES_IN_DEV)


Tuning global defaults on micro F1...
Best global: margin=0.15 minp=0.0 temp=1.5
  Micro F1=0.5698  P=0.5788  R=0.5610
Predicates in dev: ['administered', 'affect', 'change abundance', 'change effect', 'change expression', 'compared to', 'impact', 'influence', 'interact', 'is a', 'is linked to', 'located in', 'part of', 'produced by', 'strike', 'target', 'used by']


## Per-Predicate Threshold Tuning

Step 1–4: optimise margin, min_prob, max_chars, top2_gap per predicate on dev.

In [9]:
best_margin_by_pred = {}
for pred in tqdm(PREDICATES_IN_DEV, desc="Step 1 — margin"):
    best = None
    for m in PRED_MARGIN_GRID:
        pred_by_doc = {}
        for pmid, rows in dev_cache.items():
            pred_set, _ = decode_doc_pred_specific(
                rows, legal_pairs, label2id, id2label,
                BEST_MAX, BEST_MINP, BEST_MARG, 0.0, BEST_TEMP, BEST_RENORM,
                margin_by_pred={pred: m}, use_distance_aware_prob=True)
            pred_by_doc[pmid] = {t for t in pred_set if t[2] == pred}
        gold_p = {pmid: {t for t in gold_by_doc[pmid] if t[2] == pred} for pmid in gold_by_doc}
        sc = micro_scores(gold_p, pred_by_doc)
        if best is None or sc["F1"] > best["F1"]:
            best = {"margin": m, **sc}
    best_margin_by_pred[pred] = best["margin"]
print(best_margin_by_pred)


Step 1 — margin: 100%|██████████| 17/17 [01:58<00:00,  6.95s/it]

{'administered': 0.05, 'affect': 0.0, 'change abundance': 0.15, 'change effect': 0.1, 'change expression': 0.0, 'compared to': 0.0, 'impact': 0.0, 'influence': 0.25, 'interact': 0.2, 'is a': 0.1, 'is linked to': 0.0, 'located in': 0.15, 'part of': 0.25, 'produced by': 0.25, 'strike': 0.0, 'target': 0.15, 'used by': 0.25}


In [10]:
best_minprob_by_pred = {}
for pred in tqdm(PREDICATES_IN_DEV, desc="Step 2 — min_prob"):
    best = None
    for mp in PRED_MINPROB_GRID:
        pred_by_doc = {}
        for pmid, rows in dev_cache.items():
            pred_set, _ = decode_doc_pred_specific(
                rows, legal_pairs, label2id, id2label,
                BEST_MAX, BEST_MINP, BEST_MARG, 0.0, BEST_TEMP, BEST_RENORM,
                margin_by_pred=best_margin_by_pred,
                min_prob_by_pred={pred: mp}, use_distance_aware_prob=True)
            pred_by_doc[pmid] = {t for t in pred_set if t[2] == pred}
        gold_p = {pmid: {t for t in gold_by_doc[pmid] if t[2] == pred} for pmid in gold_by_doc}
        sc = micro_scores(gold_p, pred_by_doc)
        if best is None or sc["F1"] > best["F1"]:
            best = {"min_prob": mp, **sc}
    best_minprob_by_pred[pred] = best["min_prob"]
print(best_minprob_by_pred)


Step 2 — min_prob: 100%|██████████| 17/17 [01:35<00:00,  5.62s/it]

{'administered': 0.0, 'affect': 0.0, 'change abundance': 0.0, 'change effect': 0.0, 'change expression': 0.0, 'compared to': 0.0, 'impact': 0.0, 'influence': 0.0, 'interact': 0.0, 'is a': 0.0, 'is linked to': 0.0, 'located in': 0.0, 'part of': 0.0, 'produced by': 0.0, 'strike': 0.0, 'target': 0.0, 'used by': 0.0}


In [11]:
best_maxchars_by_pred = {}
for pred in tqdm(PREDICATES_IN_DEV, desc="Step 3 — max_chars"):
    best = None
    for mc in PRED_MAXCHARS_GRID:
        pred_by_doc = {}
        for pmid, rows in dev_cache.items():
            pred_set, _ = decode_doc_pred_specific(
                rows, legal_pairs, label2id, id2label,
                BEST_MAX, BEST_MINP, BEST_MARG, 0.0, BEST_TEMP, BEST_RENORM,
                margin_by_pred=best_margin_by_pred,
                min_prob_by_pred=best_minprob_by_pred,
                max_chars_by_pred={pred: mc}, use_distance_aware_prob=True)
            pred_by_doc[pmid] = {t for t in pred_set if t[2] == pred}
        gold_p = {pmid: {t for t in gold_by_doc[pmid] if t[2] == pred} for pmid in gold_by_doc}
        sc = micro_scores(gold_p, pred_by_doc)
        if best is None or sc["F1"] > best["F1"]:
            best = {"max_chars": mc, **sc}
    best_maxchars_by_pred[pred] = best["max_chars"]
print(best_maxchars_by_pred)


Step 3 — max_chars: 100%|██████████| 17/17 [02:12<00:00,  7.80s/it]

{'administered': 300, 'affect': 200, 'change abundance': 100, 'change effect': 200, 'change expression': 100, 'compared to': 100, 'impact': 200, 'influence': 250, 'interact': 150, 'is a': 100, 'is linked to': 200, 'located in': 200, 'part of': 500, 'produced by': 150, 'strike': 100, 'target': 200, 'used by': 150}


In [12]:
best_top2gap_by_pred = {}
for pred in tqdm(PREDICATES_IN_DEV, desc="Step 4 — top2_gap"):
    best = None
    for tg in PRED_TOP2_GAP_GRID:
        pred_by_doc = {}
        for pmid, rows in dev_cache.items():
            pred_set, _ = decode_doc_pred_specific(
                rows, legal_pairs, label2id, id2label,
                BEST_MAX, BEST_MINP, BEST_MARG, 0.0, BEST_TEMP, BEST_RENORM,
                margin_by_pred=best_margin_by_pred,
                min_prob_by_pred=best_minprob_by_pred,
                max_chars_by_pred=best_maxchars_by_pred,
                top2_gap_by_pred={pred: tg}, use_distance_aware_prob=True)
            pred_by_doc[pmid] = {t for t in pred_set if t[2] == pred}
        gold_p = {pmid: {t for t in gold_by_doc[pmid] if t[2] == pred} for pmid in gold_by_doc}
        sc = micro_scores(gold_p, pred_by_doc)
        if best is None or sc["F1"] > best["F1"]:
            best = {"top2_gap": tg, **sc}
    best_top2gap_by_pred[pred] = best["top2_gap"]
print(best_top2gap_by_pred)


Step 4 — top2_gap: 100%|██████████| 17/17 [01:34<00:00,  5.56s/it]

{'administered': 0.0, 'affect': 0.0, 'change abundance': 0.0, 'change effect': 0.0, 'change expression': 0.0, 'compared to': 0.0, 'impact': 0.0, 'influence': 0.0, 'interact': 0.0, 'is a': 0.0, 'is linked to': 0.0, 'located in': 0.0, 'part of': 0.0, 'produced by': 0.0, 'strike': 0.0, 'target': 0.0, 'used by': 0.0}


## Final Evaluation and Save Predictions

In [13]:
pred_by_doc_ps = {}
predictions_ps = {}

for pmid, rows in dev_cache.items():
    pred_set, best_for_pair = decode_doc_pred_specific(
        rows, legal_pairs, label2id, id2label,
        BEST_MAX, BEST_MINP, BEST_MARG, 0.0, BEST_TEMP, BEST_RENORM,
        max_chars_by_pred=best_maxchars_by_pred,
        min_prob_by_pred=best_minprob_by_pred,
        margin_by_pred=best_margin_by_pred,
        top2_gap_by_pred=best_top2gap_by_pred,
        use_distance_aware_prob=True,
    )
    pred_by_doc_ps[pmid] = pred_set
    predictions_ps[pmid] = {"mention_level_relations": build_predictions_json(best_for_pair)}

mi = micro_scores(gold_by_doc, pred_by_doc_ps)
ma = macro_scores(gold_by_doc, pred_by_doc_ps)

run_name = MODEL_DIR.name
print(f"\n*** MICRO F1 (reference metric GutBrainIE 2026): {mi['F1']:.4f} ***")
print(f"\n=== {run_name} ({POOLING_MODE}) ===")
print(f"Macro  P={ma['macro_P']:.4f}  R={ma['macro_R']:.4f}  F1={ma['macro_F1']:.4f}")
print(f"Micro  P={mi['P']:.4f}  R={mi['R']:.4f}  F1={mi['F1']:.4f}")

os.makedirs("predictions", exist_ok=True)
out_path = f"predictions/inference_{run_name}_micrf1.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(predictions_ps, f, ensure_ascii=False, indent=2)
print(f"\nSaved: {out_path}")



*** MICRO F1 (reference metric GutBrainIE 2026): 0.5871 ***

=== biolink_bert_re_A5_hardneg (mention-mean) ===
Macro  P=0.5169  R=0.5210  F1=0.5020
Micro  P=0.5978  R=0.5768  F1=0.5871

Saved: predictions/inference_biolink_bert_re_A5_hardneg_micrf1.json
